# Multi-Label Image Classification — Vision Transformer (from Scratch)

Architecture: 16×16 patch embedding → positional encoding → 6-layer Transformer encoder (8 heads, pre-LN) → CLS token → linear head.

ViTs need a lower LR and more epochs to converge from scratch; `LR=3e-4` and cosine annealing are used.

**Edit Section 7 only** when copying this notebook.

**Labels (12 classes):** `pen, paper, book, clock, phone, laptop, chair, desk, bottle, keychain, backpack, calculator`

## 1. Imports & Setup

In [ ]:
import sys
import torch
import torch.nn as nn
from pathlib import Path

sys.path.append("../")
from eval import LABEL_ORDER
from utils import (
    set_seed, load_dataset, split_dataset, subsample_subset,
    get_train_transform, get_eval_transform, build_dataloaders,
    plot_per_class_examples, plot_multilabel_examples,
    run_baselines, print_model_info,
    train_model, save_checkpoint, load_checkpoint, plot_training_history,
    collect_test_predictions, categorize_predictions,
    show_prediction_examples, plot_per_class_metrics,
    plot_confusion_matrices, plot_prediction_heatmap,
    show_saliency_examples, NUM_LABELS,
)

SEED = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


## 2. Config

In [ ]:
BASE_DATA_DIR   = "../data/aggregated"
IMAGE_SIZE      = 128
BATCH_SIZE      = 128
SPLIT           = [0.8, 0.1, 0.1]

USE_SUBSET      = False
SUBSET_FRACTION = 0.1

EXPERIMENT_NAME = "vit_scratch"
CHECKPOINT_DIR  = Path("../checkpoints")
MODEL_PATH      = CHECKPOINT_DIR / f"{EXPERIMENT_NAME}.pth"

print(f"Labels ({NUM_LABELS}): {LABEL_ORDER}")


## 3. Data Loading & Loaders

In [ ]:
full_dataset = load_dataset(BASE_DATA_DIR)
train_raw, val_raw, test_raw = split_dataset(full_dataset, SPLIT, SEED)

if USE_SUBSET:
    train_raw = subsample_subset(train_raw, SUBSET_FRACTION, SEED)
    val_raw   = subsample_subset(val_raw,   SUBSET_FRACTION, SEED + 1)
    test_raw  = subsample_subset(test_raw,  SUBSET_FRACTION, SEED + 2)

train_transform = get_train_transform(IMAGE_SIZE)
eval_transform  = get_eval_transform(IMAGE_SIZE)
train_loader, val_loader, test_loader = build_dataloaders(
    train_raw, val_raw, test_raw, train_transform, eval_transform, BATCH_SIZE
)

print(f"Total: {len(full_dataset)}  |  "
      f"Train: {len(train_raw)}  Val: {len(val_raw)}  Test: {len(test_raw)}")


## 4. Visualize Train Samples (Per Class + Multi-Label)

In [ ]:
plot_per_class_examples(train_raw)
plot_multilabel_examples(train_raw)


## 5. Baselines

In [ ]:
run_baselines(train_loader, val_loader, test_loader, NUM_LABELS, DEVICE)


## 6. Model — Vision Transformer (from Scratch)

**Edit this section** when experimenting.

`patch_size=16`, `embed_dim=256`, `depth=6`, `num_heads=8`, `mlp_dim=512` — 64 patches per 128×128 image.

In [ ]:
class VisionTransformer(nn.Module):
    """Minimal ViT: 16×16 patches → 64 tokens + CLS → 6-layer Transformer → classifier."""

    def __init__(self, num_classes, image_size=128, patch_size=16,
                 embed_dim=256, num_heads=8, depth=6, mlp_dim=512, dropout=0.1):
        super().__init__()
        assert image_size % patch_size == 0
        self.patch_size  = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        patch_dim        = 3 * patch_size * patch_size

        self.patch_embed = nn.Linear(patch_dim, embed_dim)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed   = nn.Parameter(torch.randn(1, self.num_patches + 1, embed_dim))
        self.dropout     = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=mlp_dim,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=depth)
        self.norm        = nn.LayerNorm(embed_dim)
        self.head        = nn.Sequential(nn.Dropout(dropout), nn.Linear(embed_dim, num_classes))

    def forward(self, x):
        B, C, H, W = x.shape
        ps      = self.patch_size
        patches = x.unfold(2, ps, ps).unfold(3, ps, ps)
        patches = patches.contiguous().view(B, C, -1, ps, ps)
        patches = patches.permute(0, 2, 1, 3, 4).flatten(2)   # (B, N, patch_dim)
        x       = self.patch_embed(patches)
        cls     = self.cls_token.expand(B, -1, -1)
        x       = torch.cat((cls, x), dim=1)
        x       = self.dropout(x + self.pos_embed)
        x       = self.transformer(x)
        return self.head(self.norm(x[:, 0]))                   # CLS token


def create_model(num_labels: int) -> nn.Module:
    return VisionTransformer(num_classes=num_labels, image_size=IMAGE_SIZE)


print_model_info(create_model(NUM_LABELS).to(DEVICE))


## 7. Training

In [ ]:
LR           = 3e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS   = 50

best_state, best_val_f1, history, epochs_run = train_model(
    create_model, NUM_LABELS, train_loader, val_loader, DEVICE,
    lr=LR, weight_decay=WEIGHT_DECAY, max_epochs=MAX_EPOCHS,
    warmup_epochs=10,
)

save_checkpoint(best_state, MODEL_PATH)
print(f"\nBest val F1: {best_val_f1:.4f}  |  checkpoint: {MODEL_PATH}")
plot_training_history(history, epochs_run, EXPERIMENT_NAME, LR, WEIGHT_DECAY)


## 8. Analyze Predictions

In [ ]:
model = load_checkpoint(create_model, NUM_LABELS, MODEL_PATH, DEVICE)
images, labels, preds, probs = collect_test_predictions(model, test_loader, DEVICE)
correct_idx, partial_idx, incorrect_idx = categorize_predictions(labels, preds)

show_prediction_examples(correct_idx,   images, labels, preds, "Fully Correct",     n=4)
show_prediction_examples(partial_idx,   images, labels, preds, "Partially Correct", n=4)
show_prediction_examples(incorrect_idx, images, labels, preds, "Fully Incorrect",   n=4)


In [ ]:
plot_per_class_metrics(labels, preds)
plot_confusion_matrices(labels, preds)
plot_prediction_heatmap(labels, preds)


## 9. Saliency Maps

In [ ]:
show_saliency_examples(correct_idx,   images, labels, preds, model, "Fully Correct",     n=5, device=DEVICE)
show_saliency_examples(partial_idx,   images, labels, preds, model, "Partially Correct", n=5, device=DEVICE)
show_saliency_examples(incorrect_idx, images, labels, preds, model, "Fully Incorrect",   n=5, device=DEVICE)
